In [5]:
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_community.document_loaders import PyPDFLoader
from datasets import Dataset
from ragas import evaluate
from ragas.metrics.collections import faithfulness, answer_relevancy, context_recall, context_precision
from ragas import RunConfig
import os
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter
from dotenv import load_dotenv

In [6]:
load_dotenv()

True

In [2]:
rate_limiter = InMemoryRateLimiter(
    requests_per_second=7,
    check_every_n_seconds=0.1,
    max_bucket_size=10
)

In [3]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2-preview"
)

In [7]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, rate_limiter=rate_limiter)

In [8]:
generator = TestsetGenerator(
    llm=LangchainLLMWrapper(llm),
    embedding_model=LangchainEmbeddingsWrapper(embeddings)
)

C:\Users\UserAdmin\AppData\Local\Temp\ipykernel_25204\1856198304.py:2: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  llm=LangchainLLMWrapper(llm),
C:\Users\UserAdmin\AppData\Local\Temp\ipykernel_25204\1856198304.py:3: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  embedding_model=LangchainEmbeddingsWrapper(embeddings)


In [9]:
loader = PyPDFLoader("pdfs/government-data-security-policies.pdf")
documents = loader.load()

Ignoring wrong pointing object 8 0 (offset 0)


In [10]:
testset = generator.generate_with_langchain_docs(
    documents,
    testset_size=10
)
test_df = testset.to_pandas()

Generating Samples: 100%|██████████| 12/12 [00:16<00:00,  1.38s/it]


In [11]:
test_df.to_csv("ragas_testset.csv", index=False)

In [12]:
test_df.head()

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,What is the intended audience for the Governme...,[Government Data Security Policies |  1\nGO...,The Government Data Security Policies document...,Information Security Analyst,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
1,Wht are the key policies in the Govenment's da...,[The Government takes its responsibility as a ...,The key policies in the Government's data secu...,Data Security Analyst,MISSPELLED,LONG,single_hop_specific_query_synthesizer
2,What steps should agencies take for effective ...,[Section 1: Data Security Risk Management\n01 ...,To ensure adequate and effective data security...,Information Security Analyst,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
3,How can agencies minimize the surface area of ...,[Section 3: Reduce the surface area of attack ...,Agencies can minimize the surface area of atta...,Information Security Analyst,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer
4,How do agencies log and monitor data transacti...,[<1-hop>\n\nSection 4: Log and monitor data \n...,Agencies log and monitor data transactions to ...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
